In [49]:
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import warnings
warnings.filterwarnings("ignore")

In [26]:
def relu(Z):
    return np.maximum(0, Z)


def relu_deriv(Z):
    return (Z > 0).astype(float)


def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)


def one_hot(y, num_classes=10):
    return np.eye(num_classes)[y].T

In [27]:
np.sum(np.array([[1], [2], [3], [4], [5]]), axis=1, keepdims=True)

array([[1],
       [2],
       [3],
       [4],
       [5]])

In [28]:
class NeuralNetwork:
    def __init__(self, layers, lr=0.01):
        """
        layers: list of neurons per layer, e.g. [784, 128, 64, 10]
        lr: learning rate
        """
        self.layers = layers
        self.lr = lr
        self._init_weights()

    def _init_weights(self):
        self.W1 = np.random.randn(self.layers[1], self.layers[0]) * np.sqrt(
            2.0 / self.layers[0]
        )   
        self.b1 = np.zeros((self.layers[1], 1))

        self.W2 = np.random.randn(self.layers[2], self.layers[1]) * np.sqrt(
            2.0 / self.layers[1]
        )
        self.b2 = np.zeros((self.layers[2], 1))

        self.W3 = np.random.randn(self.layers[3], self.layers[2]) * np.sqrt(
            2.0 / self.layers[2]
        )
        self.b3 = np.zeros((self.layers[3], 1))

    def forward(self, X):
        # Layer 1
        Z1 = self.W1.dot(X) + self.b1
        A1 = relu(Z1)

        # Layer 2
        Z2 = self.W2.dot(A1) + self.b2
        A2 = relu(Z2)

        # Output layer
        Z3 = self.W3.dot(A2) + self.b3
        A3 = softmax(Z3)

        cache = {"A0": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3}
        return A3, cache

    def compute_loss(self, Y, Y_hat):
        m = Y.shape[1]
        return -np.sum(Y * np.log(Y_hat + 1e-8)) / m

    def backward(self, Y, cache):
        m = Y.shape[1]

        # Output layer gradients
        dZ3 = cache["A3"] - Y
        dW3 = (1 / m) * dZ3.dot(cache["A2"].T)
        db3 = (1 / m) * np.sum(dZ3, axis=1, keepdims=True)

        # Layer 2 gradients
        dA2 = self.W3.T.dot(dZ3)
        dZ2 = dA2 * relu_deriv(cache["Z2"])
        dW2 = (1 / m) * dZ2.dot(cache["A1"].T)
        db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)

        # Layer 1 gradients
        dA1 = self.W2.T.dot(dZ2)
        dZ1 = dA1 * relu_deriv(cache["Z1"])
        dW1 = (1 / m) * dZ1.dot(cache["A0"].T)
        db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

        return dW1, db1, dW2, db2, dW3, db3

    def update(self, dW1, db1, dW2, db2, dW3, db3):
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3

    def train(self, train_loader, epochs=5):
        for epoch in range(epochs):
            total_loss, total_correct = 0, 0
            for X, y in train_loader:
                X = X.numpy().T
                y = y.numpy()
                Y = one_hot(y)
                # print(f"X: {X.shape}, Y: {Y.shape}, y: {y.shape}")

                # forward
                Y_hat, cache = self.forward(X)
                loss = self.compute_loss(Y, Y_hat)

                # backward
                dW1, db1, dW2, db2, dW3, db3 = self.backward(Y, cache)

                # update
                self.update(dW1, db1, dW2, db2, dW3, db3)

                # metrics
                total_loss += loss
                total_correct += np.sum(np.argmax(Y_hat, axis=0) == y)

            acc = total_correct / len(train_loader.dataset)
            print(
                f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Acc={acc:.4f}"
            )

    def predict(self, X):
        Y_hat, _ = self.forward(X)
        return np.argmax(Y_hat, axis=0)

In [45]:
# Load MNIST
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))]
)
train_data = datasets.MNIST(
    root="../data", train=True, download=True, transform=transform
)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


transform_test = transforms.Compose(
    [transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))]
)
test_data = datasets.MNIST(
    root="../data", train=False, download=True, transform=transform_test
)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)


In [37]:
next(enumerate(train_loader))[1][0].shape

torch.Size([64, 784])

In [31]:
# Build and Train Network
nn = NeuralNetwork(layers=[784, 128, 64, 10], lr=0.01)
nn.train(train_loader, epochs=5)

Epoch 1: Loss=0.8072, Acc=0.7908
Epoch 2: Loss=0.3554, Acc=0.8998
Epoch 3: Loss=0.2988, Acc=0.9158
Epoch 4: Loss=0.2672, Acc=0.9246
Epoch 5: Loss=0.2436, Acc=0.9313


In [56]:
# next(enumerate(test_loader))[1][0].shape 
next(enumerate(test_loader))[1][1]

tensor([7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4, 9, 6, 6, 5,
        4, 0, 7, 4, 0, 1, 3, 1, 3, 4, 7, 2, 7, 1, 2, 1, 1, 7, 4, 2, 3, 5, 1, 2,
        4, 4, 6, 3, 5, 5, 6, 0, 4, 1, 9, 5, 7, 8, 9, 3, 7, 4, 6, 4, 3, 0, 7, 0,
        2, 9, 1, 7, 3, 2, 9, 7, 7, 6, 2, 7, 8, 4, 7, 3, 6, 1, 3, 6, 9, 3, 1, 4,
        1, 7, 6, 9, 6, 0, 5, 4, 9, 9, 2, 1, 9, 4, 8, 7, 3, 9, 7, 4, 4, 4, 9, 2,
        5, 4, 7, 6, 7, 9, 0, 5, 8, 5, 6, 6, 5, 7, 8, 1, 0, 1, 6, 4, 6, 7, 3, 1,
        7, 1, 8, 2, 0, 2, 9, 9, 5, 5, 1, 5, 6, 0, 3, 4, 4, 6, 5, 4, 6, 5, 4, 5,
        1, 4, 4, 7, 2, 3, 2, 7, 1, 8, 1, 8, 1, 8, 5, 0, 8, 9, 2, 5, 0, 1, 1, 1,
        0, 9, 0, 3, 1, 6, 4, 2, 3, 6, 1, 1, 1, 3, 9, 5, 2, 9, 4, 5, 9, 3, 9, 0,
        3, 6, 5, 5, 7, 2, 2, 7, 1, 2, 8, 4, 1, 7, 3, 3, 8, 8, 7, 9, 2, 2, 4, 1,
        5, 9, 8, 7, 2, 3, 0, 4, 4, 2, 4, 1, 9, 5, 7, 7, 2, 8, 2, 6, 8, 5, 7, 7,
        9, 1, 8, 1, 8, 0, 3, 0, 1, 9, 9, 4, 1, 8, 2, 1, 2, 9, 7, 5, 9, 2, 6, 4,
        1, 5, 8, 2, 9, 2, 0, 4, 0, 0, 2,

In [ ]:
predictions = nn.predict(next(enumerate(test_loader))[1][0].T)

(1000,)

In [62]:
np.sum((predictions == next(enumerate(test_loader))[1][1]).numpy())

np.int64(926)